# Geospatial Pipeline — Development Notebook
**Project:** MO Voter Resource Allocation  
**Purpose:** Interactive development environment for Stages 1–7 of the
precinct-level geospatial pipeline. Once each stage is working here, the
logic lives in `src/census_block_loader.py` and `src/precinct_builder.py`.
Run `main.py` to execute the full pipeline in production.

---

## Stage 0 — Configuration & Imports

In [ ]:
import os
import sys
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Make sure src/ modules are importable from this notebook
sys.path.insert(0, os.path.abspath(".."))

from src.geo_loader         import GeoLoader
from src.census_block_loader import CensusBlockLoader
from src.precinct_builder   import (
    align_crs,
    apportion_blocks_to_precincts,
    extract_vest_vote_totals,
    calculate_turnout,
    apportion_acs_to_precincts,
    aggregate_to_county,
    build_precinct_features,
)

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_RAW       = "../data/raw/"
DATA_PROCESSED = "../data/processed/"
GEO_RAW        = "../data/geo/raw/"
GEO_PROCESSED  = "../data/geo/processed/"

# ── Census API key ─────────────────────────────────────────────────────────
# Free key: https://api.census.gov/data/key_signup.html
CENSUS_API_KEY = "YOUR_API_KEY_HERE"   # <-- fill this in before running Stage 2

print("Imports OK.")

## Stage 1 — Load VEST Precinct Shapefiles

Each VEST shapefile is both the **geometry** (precinct boundaries) and the
**election data** (vote counts per candidate, embedded in the attribute table).

We load all three years and inspect their columns to confirm the naming
convention before doing anything with them.

In [ ]:
geo_loader = GeoLoader(geo_raw_dir=GEO_RAW)

vest = {}
for year in [2016, 2020, 2024]:
    print(f"Loading {year} VEST shapefile...")
    vest[year] = geo_loader.get_precinct_shapefile(year)
    print(f"  {len(vest[year]):,} precincts | CRS: {vest[year].crs}")
    print()

# Quick look at 2016 columns — all three should follow the same G{YY}PRE pattern
print("2016 columns (first 20):")
print(list(vest[2016].columns)[:20])

In [ ]:
# Spot-check: confirm presidential vote columns exist for each year
import re
for year in [2016, 2020, 2024]:
    suffix = str(year)[2:]
    pre_cols = [c for c in vest[year].columns if re.match(rf"^G{suffix}PRE[A-Z]+$", c)]
    print(f"{year}: {len(pre_cols)} presidential candidate columns → {pre_cols[:5]}")

In [ ]:
# Quick map — sanity check that geometries look right
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, year in zip(axes, [2016, 2020, 2024]):
    vest[year].plot(ax=ax, edgecolor="black", linewidth=0.1, color="lightblue")
    ax.set_title(f"MO Precincts — {year}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Stage 2 — Load Census Block Data

Census blocks are the atomic geographic unit we use to bridge between
county-level ACS demographics and precinct-level analysis.

**First run:** Downloads ~60MB TIGER/Line shapefile from Census Bureau (one-time).  
**Population data:** Requires a Census API key set above.  
**Subsequent runs:** Loads from cached local files instantly.

In [ ]:
block_loader = CensusBlockLoader(
    geo_raw_dir=GEO_RAW,
    census_api_key=CENSUS_API_KEY
)

# This will download on first run, then load from cache
blocks_gdf = block_loader.get_blocks_with_population()

print(f"Blocks loaded: {len(blocks_gdf):,}")
print(f"CRS: {blocks_gdf.crs}")
print(f"Total MO population: {blocks_gdf['total_population'].sum():,.0f}")
print(f"Total MO VAP: {blocks_gdf['vap_total'].sum():,.0f}")
blocks_gdf.head()

## Stage 3 — CRS Alignment

Before any spatial join, all layers must be in the same projected CRS.
We use EPSG:5070 (Albers Equal Area Conus) — the same CRS used elsewhere
in our pipeline — because it preserves area, which matters for apportionment.

In [ ]:
# Align all layers to the same CRS
for year in [2016, 2020, 2024]:
    vest[year] = align_crs(vest[year])

blocks_aligned = align_crs(blocks_gdf)

# Confirm
print("CRS after alignment:")
for year in [2016, 2020, 2024]:
    print(f"  VEST {year}: {vest[year].crs.to_epsg()}")
print(f"  Blocks:    {blocks_aligned.crs.to_epsg()}")

## Stage 4 — Detect Precinct ID Column

The column that uniquely identifies each precinct varies across VEST files.
We need to identify it before running the apportionment.

In [ ]:
# Inspect candidate ID columns in each VEST file
# Common candidates: GEOID20, VTDST20, precinct_id, NAME
for year in [2016, 2020, 2024]:
    print(f"\n{year} columns containing 'id', 'geo', or 'vtd':")
    candidate_cols = [c for c in vest[year].columns
                      if any(k in c.lower() for k in ["id", "geo", "vtd", "name", "prec"])]
    print(f"  {candidate_cols}")

In [ ]:
# Set your precinct ID column after inspecting above
# TODO: update this after running the cell above
PRECINCT_ID_COL = "GEOID20"   # <-- update if different in your VEST files

# Verify it uniquely identifies precincts (no duplicates)
for year in [2016, 2020, 2024]:
    if PRECINCT_ID_COL in vest[year].columns:
        n_precincts = len(vest[year])
        n_unique    = vest[year][PRECINCT_ID_COL].nunique()
        print(f"{year}: {n_precincts:,} rows, {n_unique:,} unique {PRECINCT_ID_COL} values")
        if n_precincts != n_unique:
            print(f"  ⚠️  WARNING: {PRECINCT_ID_COL} is not unique — review before joining.")
    else:
        print(f"{year}: Column '{PRECINCT_ID_COL}' not found.")

## Stage 5 — Apportionment: Census Blocks → Precincts

This is the core spatial operation. For each precinct, we calculate what
fraction of each overlapping census block falls inside it, then apportion
that block's population proportionally.

**Expected runtime:** 1–3 minutes per year (geopandas overlay on ~228k blocks).

Start with 2020 to validate before running all years.

In [ ]:
# Run apportionment for 2020 first as a test
pop_2020 = apportion_blocks_to_precincts(
    blocks_gdf  = blocks_aligned,
    precincts_gdf = vest[2020],
    precinct_id_col = PRECINCT_ID_COL
)

print(f"\nApportionment result: {len(pop_2020):,} precincts")
print(f"Total apportioned population: {pop_2020['apportioned_population'].sum():,.0f}")
print(f"Total apportioned VAP:        {pop_2020['apportioned_vap'].sum():,.0f}")
pop_2020.head()

In [ ]:
# QA: any precincts with zero population?
zero_pop = (pop_2020["apportioned_population"] == 0).sum()
zero_vap = (pop_2020["apportioned_vap"] == 0).sum()
print(f"Precincts with 0 population: {zero_pop}")
print(f"Precincts with 0 VAP:        {zero_vap}")
# A small number of zero-population precincts is normal (e.g. industrial areas).
# A large number indicates a geometry mismatch — check CRS alignment.

## Stage 6 — Extract Vote Totals and Calculate Turnout

In [ ]:
# Extract presidential votes for 2020
votes_2020 = extract_vest_vote_totals(vest[2020], year=2020,
                                       precinct_id_col=PRECINCT_ID_COL)
print(f"Total 2020 presidential votes: {votes_2020['total_votes'].sum():,.0f}")
votes_2020.head()

In [ ]:
# Calculate turnout
turnout_2020 = calculate_turnout(votes_2020, pop_2020, PRECINCT_ID_COL)

print(f"Median precinct turnout: {turnout_2020['turnout_pct'].median():.1f}%")
print(f"Precincts with turnout > 100%: {(turnout_2020['turnout_pct'] > 100).sum()}")
print(f"Precincts with NaN turnout:    {turnout_2020['turnout_pct'].isna().sum()}")

turnout_2020[["total_votes", "apportioned_vap", "turnout_pct", "rep_pct", "dem_pct"]].describe()

## Stage 7 — Attach ACS Demographics

Load the staged ACS CSVs and downscale county-level values to precincts
using population weighting.

In [ ]:
# Load staged ACS files
acs_categories = ["income", "education", "race", "commute", "sex_age"]
acs_staging = {
    cat: pd.read_csv(f"{DATA_PROCESSED}stg_census_{cat}.csv")
    for cat in acs_categories
}

for cat, df in acs_staging.items():
    print(f"{cat}: {len(df)} rows, years: {sorted(df['census_year'].unique())}")

In [ ]:
# Build full precinct feature set for 2020
precinct_2020 = build_precinct_features(
    vest_gdf        = vest[2020],
    blocks_gdf      = blocks_aligned,
    acs_staging_dfs = acs_staging,
    year            = 2020,
    precinct_id_col = PRECINCT_ID_COL
)

print(f"\nFinal precinct GDF: {len(precinct_2020):,} rows × {len(precinct_2020.columns)} columns")
print("Columns:", list(precinct_2020.columns))

## Stage 8 — County Aggregation

Roll up precinct data to county level for the second map layer.
This uses the same precinct data — no new joins needed.

In [ ]:
county_2020 = aggregate_to_county(precinct_2020, PRECINCT_ID_COL)

print(f"County GDF: {len(county_2020)} counties")
county_2020[["turnout_pct", "rep_pct", "dem_pct"]].describe()

## Stage 9 — Quick Visualisation (Sanity Check)

Two-layer view: county risk classification + precinct turnout breakdown.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# County layer — turnout classification
county_2020.plot(
    column="turnout_pct", cmap="RdYlGn", legend=True,
    edgecolor="white", linewidth=0.5, ax=axes[0],
    missing_kwds={"color": "lightgrey"}
)
axes[0].set_title("2020 County-Level Turnout Rate", fontsize=14)
axes[0].axis("off")

# Precinct layer — granular turnout
precinct_2020.plot(
    column="turnout_pct", cmap="RdYlGn", legend=True,
    edgecolor="none", ax=axes[1],
    missing_kwds={"color": "lightgrey"}
)
axes[1].set_title("2020 Precinct-Level Turnout Rate", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Stage 10 — Multi-Year Run and Export

Once 2020 is validated, run all three years and save outputs.

In [ ]:
# TODO: Run once 2020 QA is complete
# all_years = {}
# for year in [2016, 2020, 2024]:
#     all_years[year] = build_precinct_features(
#         vest_gdf        = vest[year],
#         blocks_gdf      = blocks_aligned,
#         acs_staging_dfs = acs_staging,
#         year            = year,
#         precinct_id_col = PRECINCT_ID_COL
#     )
#
# # Combine all years into one flat DataFrame for Random Forest
# combined_df = pd.concat(
#     [gdf.drop(columns="geometry") for gdf in all_years.values()],
#     ignore_index=True
# )
# combined_df.to_csv(f"{DATA_PROCESSED}stg_precinct_features_all_years.csv", index=False)
# print(f"Saved: {len(combined_df):,} rows × {len(combined_df.columns)} columns")

In [ ]:
# Save 2020 outputs once validated
# Precinct GeoJSON → precinct map layer
out_precinct = f"{GEO_PROCESSED}precinct_features_2020.geojson"
precinct_2020.to_file(out_precinct, driver="GeoJSON")
print(f"Saved precinct layer: {out_precinct}")

# County GeoJSON → county map layer
out_county = f"{GEO_PROCESSED}county_features_2020.geojson"
county_2020.to_file(out_county, driver="GeoJSON")
print(f"Saved county layer: {out_county}")

# Flat CSV → modeling input
out_csv = f"{DATA_PROCESSED}stg_precinct_features_2020.csv"
precinct_2020.drop(columns="geometry").to_csv(out_csv, index=False)
print(f"Saved modeling table: {out_csv}")